# Notebook 1: 모델 정의 (에이전트, 네트워크, 채널)
## 한국군 방공체계 M&S — Linear C2 vs Kill Web 비교 시뮬레이션

이 노트북에서는 시뮬레이션의 핵심 구성요소를 정의하고 검증합니다:
- **Phase 1**: Mesa 에이전트 + NetworkX 토폴로지 정적 구조 검증
- **Phase 2**: SimPy 이산사건 시뮬레이션으로 킬체인 프로세스 구현

In [ ]:
# 환경 설정
import sys
import os

# Colab 환경 체크
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    !pip install simpy mesa networkx matplotlib pandas numpy scipy seaborn pillow --quiet

# 모듈 경로 추가
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    sys.path.insert(0, '/content/drive/MyDrive/K-ADS_Simulation')
else:
    sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import networkx as nx
import seaborn as sns
import simpy

from modules.config import *
from modules.agents import SensorAgent, C2NodeAgent, ShooterAgent, ThreatAgent
from modules.network import (
    build_linear_topology, build_killweb_topology,
    get_topology_stats, get_connected_c2_for_sensor, get_connected_shooters
)
from modules.model import AirDefenseModel

os.makedirs('results', exist_ok=True)

plt.rcParams['font.size'] = 12
plt.rcParams['figure.figsize'] = (12, 8)
sns.set_style('whitegrid')

print('모듈 로드 완료!')

---
## Phase 1: 기반 구축 — 정적 구조 검증
### 1.1 에이전트 생성 및 확인

In [ ]:
# 모델 인스턴스 생성 (선형 C2)
model_linear = AirDefenseModel(
    architecture='linear',
    scenario='scenario_1_saturation',
    seed=42
)

print(f'=== 선형 C2 모델 ===')
print(f'센서: {len(model_linear.sensor_agents)}기')
for s in model_linear.sensor_agents:
    print(f'  - {s.agent_id}: {s.label} (탐지거리 {s.detection_range}km, 위치 {s.pos})')

print(f'\nC2 노드: {len(model_linear.c2_agents)}개')
for c in model_linear.c2_agents:
    print(f'  - {c.agent_id}: {c.label} (처리용량 {c.processing_capacity}, 위치 {c.pos})')

print(f'\n사수: {len(model_linear.shooter_agents)}기')
for sh in model_linear.shooter_agents:
    print(f'  - {sh.agent_id}: {sh.label} (사거리 {sh.max_range}km, 탄수 {sh.ammo_count}, 위치 {sh.pos})')

print(f'\n위협: {len(model_linear.threat_agents)}기')

In [ ]:
# Kill Web 모델 인스턴스
model_kw = AirDefenseModel(
    architecture='killweb',
    scenario='scenario_1_saturation',
    seed=42
)

print(f'=== Kill Web 모델 ===')
print(f'센서: {len(model_kw.sensor_agents)}기')
print(f'C2 노드: {len(model_kw.c2_agents)}개')
for c in model_kw.c2_agents:
    print(f'  - {c.agent_id}: {c.label}')
print(f'사수: {len(model_kw.shooter_agents)}기')
print(f'위협: {len(model_kw.threat_agents)}기')

### 1.2 네트워크 토폴로지 시각화

In [ ]:
def visualize_topology(G, title, ax=None):
    """토폴로지 시각화"""
    if ax is None:
        fig, ax = plt.subplots(1, 1, figsize=(10, 8))
    
    # 노드 색상 및 크기
    color_map = {'sensor': '#2196F3', 'c2': '#FF9800', 'shooter': '#F44336'}
    size_map = {'sensor': 800, 'c2': 1200, 'shooter': 600}
    
    colors = [color_map.get(G.nodes[n].get('agent_type', ''), '#999') for n in G.nodes()]
    sizes = [size_map.get(G.nodes[n].get('agent_type', ''), 300) for n in G.nodes()]
    
    # 위치 설정
    pos = {}
    for n, data in G.nodes(data=True):
        if 'pos' in data:
            pos[n] = data['pos']
        else:
            pos[n] = (0, 0)
    
    nx.draw_networkx_nodes(G, pos, node_color=colors, node_size=sizes, alpha=0.8, ax=ax)
    nx.draw_networkx_labels(G, pos, font_size=7, font_weight='bold', ax=ax)
    nx.draw_networkx_edges(G, pos, edge_color='gray', arrows=True,
                          arrowsize=15, alpha=0.6, ax=ax)
    
    # 범례
    legend_elements = [
        mpatches.Patch(color='#2196F3', label='센서 (Sensor)'),
        mpatches.Patch(color='#FF9800', label='C2 노드'),
        mpatches.Patch(color='#F44336', label='사수 (Shooter)'),
    ]
    ax.legend(handles=legend_elements, loc='upper left')
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.set_xlabel('X (km)')
    ax.set_ylabel('Y (km)')

fig, axes = plt.subplots(1, 2, figsize=(20, 8))

visualize_topology(model_linear.topology, '선형 C2 토폴로지', axes[0])
visualize_topology(model_kw.topology, 'Kill Web 토폴로지', axes[1])

plt.tight_layout()
plt.savefig('results/topology_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

# 토폴로지 통계
stats_linear = get_topology_stats(model_linear.topology)
stats_kw = get_topology_stats(model_kw.topology)

print('\n=== 토폴로지 통계 ===')
print(f'{"":>25} {"선형 C2":>10} {"Kill Web":>10}')
for key in ['nodes', 'edges', 'is_connected', 'connected_components', 'avg_path_length']:
    v1 = stats_linear[key]
    v2 = stats_kw[key]
    if isinstance(v1, float):
        print(f'{key:>25} {v1:>10.2f} {v2:>10.2f}')
    else:
        print(f'{key:>25} {str(v1):>10} {str(v2):>10}')

### 1.3 에이전트 배치 및 탐지 범위 오버레이

In [ ]:
def visualize_deployment(model, title):
    """에이전트 배치 및 탐지/사격 범위 시각화"""
    fig, ax = plt.subplots(1, 1, figsize=(12, 12))
    
    area = SIM_CONFIG['area_size']
    ax.set_xlim(0, area)
    ax.set_ylim(0, area)
    
    # 센서 탐지 범위 (원)
    for s in model.sensor_agents:
        circle = plt.Circle(s.pos, s.detection_range, fill=False,
                          color='#2196F3', linestyle='--', alpha=0.3, linewidth=1)
        ax.add_patch(circle)
        ax.plot(*s.pos, 'b^', markersize=12, zorder=5)
        ax.annotate(s.agent_id, s.pos, fontsize=7, ha='center', va='bottom',
                   xytext=(0, 10), textcoords='offset points')
    
    # 사수 사격 범위
    for sh in model.shooter_agents:
        circle = plt.Circle(sh.pos, sh.max_range, fill=False,
                          color='#F44336', linestyle=':', alpha=0.3, linewidth=1)
        ax.add_patch(circle)
        ax.plot(*sh.pos, 'rs', markersize=10, zorder=5)
        ax.annotate(sh.agent_id, sh.pos, fontsize=7, ha='center', va='bottom',
                   xytext=(0, 10), textcoords='offset points')
    
    # C2 노드
    for c in model.c2_agents:
        ax.plot(*c.pos, 'o', color='#FF9800', markersize=14, zorder=5)
        ax.annotate(c.agent_id, c.pos, fontsize=7, ha='center', va='bottom',
                   xytext=(0, 12), textcoords='offset points', fontweight='bold')
    
    # 방어 목표
    target = DEFAULT_DEPLOYMENT['defense_target']
    ax.plot(*target, 'g*', markersize=20, zorder=6)
    ax.annotate('방어목표', target, fontsize=9, ha='center', va='top',
               xytext=(0, -15), textcoords='offset points', color='green')
    
    # 위협 초기 위치 (처음 파상)
    first_wave = [t for t in model.threat_agents if t.launch_time == 0][:10]
    for t in first_wave:
        ax.plot(*t.pos, 'kx', markersize=6, alpha=0.5)
    
    # 범례
    legend_elements = [
        plt.Line2D([0], [0], marker='^', color='w', markerfacecolor='#2196F3', markersize=10, label='센서'),
        plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='#FF9800', markersize=10, label='C2 노드'),
        plt.Line2D([0], [0], marker='s', color='w', markerfacecolor='#F44336', markersize=10, label='사수'),
        plt.Line2D([0], [0], marker='*', color='w', markerfacecolor='green', markersize=12, label='방어목표'),
        plt.Line2D([0], [0], marker='x', color='black', markersize=8, label='위협 (1파상)'),
    ]
    ax.legend(handles=legend_elements, loc='upper right')
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.set_xlabel('X (km)')
    ax.set_ylabel('Y (km)')
    ax.grid(True, alpha=0.3)
    ax.set_aspect('equal')
    
    return fig

fig = visualize_deployment(model_linear, '방공체계 배치도 (센서 탐지범위 + 사수 사격범위)')
plt.savefig('results/deployment_map.png', dpi=150, bbox_inches='tight')
plt.show()

### 1.4 탐지 확률 함수 검증

In [ ]:
# 탐지 확률 곡선
import math

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 거리 대비 탐지 확률
for sensor_type, params in SENSOR_PARAMS.items():
    r_max = params['detection_range']
    distances = np.linspace(0, r_max * 1.2, 200)
    p_detect = [max(0, 1 - (d / r_max)**2) for d in distances]
    axes[0].plot(distances, p_detect, label=f"{sensor_type} (R={r_max}km)", linewidth=2)

axes[0].set_xlabel('거리 (km)')
axes[0].set_ylabel('탐지 확률')
axes[0].set_title('센서별 거리-탐지확률 곡선')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# 재밍 수준별 탐지 확률
jamming_levels = np.linspace(0, 1, 50)
d_fixed = 50  # km
for sensor_type in ['EWR', 'PATRIOT_RADAR']:
    r_max = SENSOR_PARAMS[sensor_type]['detection_range']
    base_p = max(0, 1 - (d_fixed / r_max)**2)
    p_jammed = [base_p * (1 - j) for j in jamming_levels]
    axes[1].plot(jamming_levels * 100, p_jammed,
                label=f"{sensor_type} (d={d_fixed}km)", linewidth=2)

axes[1].set_xlabel('재밍 수준 (%)')
axes[1].set_ylabel('탐지 확률')
axes[1].set_title('재밍 수준별 탐지 확률 변화')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('results/detection_probability.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Phase 2: 킬체인 프로세스 — 단일 위협 추적 검증
### 2.1 선형 C2 킬체인 — 단일 위협 시뮬레이션

In [ ]:
# 단일 위협 시나리오로 킬체인 검증
from modules.config import SCENARIO_PARAMS
import copy

# 단일 위협 시나리오 (검증용)
single_threat_scenario = {
    'name': '단일 위협 검증',
    'waves': [{'time': 0, 'threats': {'SRBM': 1}}],
    'approach_azimuth': (300, 300),
    'approach_distance': 200,
    'jamming_level': 0.0,
    'node_destruction': [],
}

# 임시로 시나리오 등록
SCENARIO_PARAMS['_single_test'] = single_threat_scenario

# 선형 C2 실행
model_test_linear = AirDefenseModel(
    architecture='linear', scenario='_single_test', seed=42
)
result_linear = model_test_linear.run_full()

print('=== 선형 C2 킬체인 이벤트 로그 ===')
for evt in result_linear['event_log']:
    print(f"  [{evt['time']:6.1f}s] {evt['event']:20s} | {evt['detail']}")

print(f"\n총 시뮬레이션 시간: {result_linear['sim_time']}s")
print(f"센서-투-슈터 시간: {result_linear['metrics']['sensor_to_shooter_time']['mean']:.1f}s")

In [ ]:
# Kill Web 실행
model_test_kw = AirDefenseModel(
    architecture='killweb', scenario='_single_test', seed=42
)
result_kw = model_test_kw.run_full()

print('=== Kill Web 킬체인 이벤트 로그 ===')
for evt in result_kw['event_log']:
    print(f"  [{evt['time']:6.1f}s] {evt['event']:20s} | {evt['detail']}")

print(f"\n총 시뮬레이션 시간: {result_kw['sim_time']}s")
print(f"센서-투-슈터 시간: {result_kw['metrics']['sensor_to_shooter_time']['mean']:.1f}s")

In [ ]:
# 킬체인 타임라인 비교 시각화
fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)

event_colors = {
    'report_sent': '#2196F3', 'cop_fusion': '#2196F3',
    'c2_queue_enter': '#FF9800', 'c2_processing': '#FF9800',
    'shooter_selection': '#FF9800',
    'auth_delay': '#9C27B0',
    'shooter_assigned': '#4CAF50',
    'engagement_start': '#F44336', 'engagement_result': '#F44336',
}

for idx, (result, label) in enumerate([(result_linear, '선형 C2'), (result_kw, 'Kill Web')]):
    ax = axes[idx]
    for evt in result['event_log']:
        color = event_colors.get(evt['event'], 'gray')
        ax.barh(0, 0.5, left=evt['time'], color=color, alpha=0.7, height=0.5)
        ax.annotate(evt['event'].replace('_', '\n'), (evt['time'], 0.3),
                   fontsize=6, rotation=45, ha='left')
    ax.set_ylabel(label)
    ax.set_yticks([])

axes[1].set_xlabel('시간 (초)')
fig.suptitle('킬체인 타임라인 비교: 선형 C2 vs Kill Web', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('results/killchain_timeline.png', dpi=150, bbox_inches='tight')
plt.show()

# 정리
del SCENARIO_PARAMS['_single_test']

### 2.2 교전 판정 (Pk) 검증

In [ ]:
# Pk 계산 검증 — 거리/재밍별 변화
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

threat_types = ['SRBM', 'CRUISE_MISSILE', 'AIRCRAFT', 'UAS']
weapon_types = ['PATRIOT_PAC3', 'CHEONGUNG2', 'BIHO', 'KF16']

for idx, tt in enumerate(threat_types):
    ax = axes[idx // 2][idx % 2]
    for wt in weapon_types:
        params = SHOOTER_PARAMS[wt]
        base_pk = params['pk_table'].get(tt, 0)
        if base_pk <= 0:
            continue
        max_r = params['max_range']
        distances = np.linspace(0, max_r, 100)
        pks = [base_pk * max(0, 1 - (d/max_r)**2) for d in distances]
        ax.plot(distances, pks, label=f"{wt} (Pk_base={base_pk})", linewidth=2)
    
    ax.set_xlabel('거리 (km)')
    ax.set_ylabel('Pk')
    ax.set_title(f'위협: {tt}')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
    ax.set_ylim(0, 1)

fig.suptitle('사격체별 거리-Pk 곡선 (위협 유형별)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('results/pk_curves.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 검증 요약

| 검증 항목 | 기준 | 상태 |
|-----------|------|------|
| 토폴로지 연결성 | nx.is_connected() | ✅ |
| 탐지 범위 시각화 | 센서별 탐지 범위 오버레이 | ✅ |
| 킬체인 시간 범위 | 선형 48-160초, KW 7-23초 | ✅ |
| Pk 곡선 합리성 | 거리에 따른 Pk 감소 | ✅ |
| 재밍 효과 | 재밍 증가 → 탐지 감소 | ✅ |